# 过滤后的 Liu2023 全量消融图

绘制三个版本的联合 UMAP，及类型合并和按 Subclass 分面组成的边缘对齐图。

In [ ]:
from pathlib import Path
from datetime import datetime,timezone,timedelta
import os,sys,json,hashlib,gc,shutil
import numpy as np
import pandas as pd
import anndata as ad
from threadpoolctl import threadpool_limits

from data.paths import PROJECT, OUTPUTS
OUTPUTS=OUTPUTS/'min10'
ROOT=OUTPUTS/'ablation/liu2023/full'
PREP=OUTPUTS/'preprocessing/liu2023/full'
os.chdir(PROJECT)
sys.path.insert(0,str(PROJECT))
assert Path(sys.prefix).name.lower()=='simscdnam'
verification=json.loads((ROOT/'verification.json').read_text())
assert verification['状态']=='通过'
aliases={'同时去掉掩码和对齐':'Raw generator','去掉对齐，保留掩码':'Mask only','完整simscDNAm':'Full model'}
METHODS=list(aliases.values())
paths={aliases[name]:Path(path) for name,path in verification['版本矩阵'].items()}
paths={'真实数据':PREP/'HeldoutReal.h5ad',**paths}
scores=pd.read_csv(ROOT/'scores.csv')
score_hash=hashlib.sha256((ROOT/'scores.csv').read_bytes()).hexdigest()
stamps={str(path):{'size':path.stat().st_size,'mtime_ns':path.stat().st_mtime_ns} for path in [*paths.values(),PREP/'TrainingReal.h5ad']}

def visual_status(phase,**extra):
    record={'阶段':phase,'时间':datetime.now(timezone(timedelta(hours=8))).isoformat(),**extra}
    partial=ROOT/'visual_status.json.partial'
    partial.write_text(json.dumps(record,ensure_ascii=False,indent=2))
    partial.replace(ROOT/'visual_status.json')
    print(record,flush=True)

from evaluate.plotting import PLOT_COLORS
PLOT_COLORS.update({'Raw generator':'#CC79A7','Mask only':'#E69F00','Full model':'#0072B2'})
visual_status('绘制消融边缘分布')


In [ ]:
from evaluate.A1.plotting import load_plot_regions,plot_a1
from data.evaluation import read_labels,validate_region_axis

datasets={name:ad.read_h5ad(path,backed='r') for name,path in paths.items()}
try:
    real=datasets['真实数据']
    expected=real.obs['Subclass'].astype(str).value_counts().sort_index()
    rows={name:np.arange(item.n_obs) for name,item in datasets.items()}
    for name,item in datasets.items():
        validate_region_axis(real,item)
        assert item.n_obs==real.n_obs
        pd.testing.assert_series_equal(item.obs['Subclass'].astype(str).value_counts().sort_index(),expected,check_names=False)
    regions=load_plot_regions(real,real.n_vars-1000,PROJECT/'evaluate/A1/marker_regions_liu2023.csv','mm10')
    plot_a1(datasets,rows,METHODS,regions,ROOT)
finally:
    for item in datasets.values():item.file.close()
for name,keys in [('A1_plot_histograms.csv',['原region','方法','分箱左界']),('A1_type_histograms.csv',['原region','细胞类型','方法','分箱左界'])]:
    original=pd.read_csv(OUTPUTS/'internal/jobs/liu2023/full/simscDNAm'/name)
    original=original.loc[original['方法'].isin(['真实数据','simscDNAm'])]
    current=pd.read_csv(ROOT/name)
    current=current.loc[current['方法'].isin(['真实数据','Full model'])].copy()
    current['方法']=current['方法'].replace({'Full model':'simscDNAm'})
    pd.testing.assert_frame_equal(current.sort_values(keys).reset_index(drop=True),original.sort_values(keys).reset_index(drop=True),check_exact=True)
visual_status('两张边缘图完成；完整模型分布源表与主实验逐值一致')


In [ ]:
from data.evaluation import imputed_features
from evaluate.A3.metrics import milisi
from sklearn.decomposition import PCA
import umap

visual_status('拟合共享训练 PCA 并核对三版本 miLISI')
with threadpool_limits(limits=8):
    with np.load(OUTPUTS/'internal/jobs/liu2023/full/pca.npz') as cache:
        selected,fill=cache['selected'],cache['fill']
        pca=PCA(n_components=len(cache['components']),svd_solver='randomized',random_state=42)
        pca.components_=cache['components']
        pca.mean_=cache['mean']
        pca.n_features_in_=len(selected)
    representations={}
    labels={}
    identifiers={}
    for name,path in paths.items():
        item=ad.read_h5ad(path)
        labels[name]=read_labels(item,np.arange(item.n_obs))
        identifiers[name]=item.obs_names.to_numpy(dtype=str)
        representations[name]=pca.transform(imputed_features(item,np.arange(item.n_obs),selected,fill))
        del item
        gc.collect()
    summary=scores.rename(columns={'A3 miLISI↑':'miLISI'}).copy()
    summary['方法']=summary['方法'].map(aliases)
    summary['近邻数']=30
    checks=[]
    for method in METHODS:
        score,_,_=milisi(representations['真实数据'],labels['真实数据'],representations[method],labels[method],30)
        expected_score=float(summary.loc[summary['方法'].eq(method),'miLISI'].iloc[0])
        np.testing.assert_allclose(score,expected_score,rtol=0,atol=1e-12)
        checks.append({'方法':method,'重新计算miLISI':score,'消融表miLISI':expected_score})
    pd.DataFrame(checks).to_csv(ROOT/'visual_milisi_checks.csv',index=False)
    visual_status('三版本 miLISI 与消融表一致；计算 UMAP')
    frames=[]
    for method in METHODS:
        combined=np.vstack([representations['真实数据'],representations[method]])
        xy=umap.UMAP(n_neighbors=30,random_state=2022,transform_seed=2022,n_jobs=1).fit_transform(combined)
        frames.append(pd.DataFrame({'方法':method,'来源':np.repeat(['真实','合成'],[len(identifiers['真实数据']), len(identifiers[method])]),
            '细胞ID':np.r_[identifiers['真实数据'],identifiers[method]],'UMAP1':xy[:,0],'UMAP2':xy[:,1]}))
        visual_status('UMAP 版本完成',版本=method)
coordinates=pd.concat(frames,ignore_index=True)
coordinates.to_csv(ROOT/'A3_joint_coordinates.csv',index=False)
from evaluate.A3.plotting import plot_joint_umap
plot_joint_umap(coordinates,summary,METHODS,ROOT,2022)


In [ ]:
from PIL import Image
from pypdf import PdfReader, PdfWriter, PageObject, Transformation

figures=OUTPUTS/'figures'
figures.mkdir(parents=True,exist_ok=True)
for extension in ('png','pdf'):
    shutil.copy2(ROOT/'figures'/f'A3_joint_UMAP.{extension}',figures/f'Ablation_A3_joint_UMAP.{extension}')

stems=('A1_marginal_regions','A1_region_by_type')
panels=[]
for stem in stems:
    with Image.open(ROOT/'figures'/f'{stem}.png') as source:
        panel=source.convert('RGB')
        height=round(panel.height*4000/panel.width)
        panels.append(panel.resize((4000,height),Image.Resampling.LANCZOS))
canvas=Image.new('RGB',(4000,sum(panel.height for panel in panels)+48),'white')
canvas.paste(panels[0],(0,0))
canvas.paste(panels[1],(0,panels[0].height+48))
canvas.save(figures/'Ablation_marginal_alignment.png',dpi=(300,300))

pages=[PdfReader(ROOT/'figures'/f'{stem}.pdf').pages[0] for stem in stems]
width=max(float(page.mediabox.width) for page in pages)
scales=[width/float(page.mediabox.width) for page in pages]
heights=[float(page.mediabox.height)*scale for page,scale in zip(pages,scales)]
top=sum(heights)+12
sheet=PageObject.create_blank_page(width=width,height=top)
for page,scale,height in zip(pages,scales,heights):
    sheet.merge_transformed_page(page,Transformation().scale(scale).translate(0,top-height))
    top-=height+12
writer=PdfWriter()
writer.add_page(sheet)
writer.write(figures/'Ablation_marginal_alignment.pdf')
for path,stamp in stamps.items():
    assert Path(path).stat().st_size==stamp['size'] and Path(path).stat().st_mtime_ns==stamp['mtime_ns'],path
assert hashlib.sha256((ROOT/'scores.csv').read_bytes()).hexdigest()==score_hash
result={'状态':'通过','图像数':2,'版本数':len(METHODS),'每版本细胞数':len(identifiers['真实数据']),
    '逐类型配额与区域轴一致':True,'完整模型边缘分布源表与主实验一致':True,'三版本miLISI与消融表一致':True}
temporary=(ROOT/'visual_verification.json.partial')
temporary.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
temporary.replace(ROOT/'visual_verification.json')
visual_status('完成',**result)